# Kryptografia asymetryczna - kryptosystem RSA 
Kryptografia asymetryczna charakteryzuje się wykorzystaniem **pary kluczy publiczny-prywatny** (stąd nazwa kryptografia z kluczem publicznym). Klucz publiczny może być swobodnie dystrybuowany otwartym kanałem i służy do szyfrowania (a także do weryfikowania podpisu). Klucz prywatny musi być utrzymywany w tajności i służy do deszyfrowania (lub tworzenia podpisu). 

Chronologicznie pierwszym kryptosystemem asymetrycznym był protokół wymiany kluczu Diffiego-Hellmana-Merkla. Służy on bezpiecznej wymiany danych, które mogą być wykorzystane jako tajne klucze kryptograficzne lub mogą być użyte do wyprodukowania kluczy. 

Najbardziej znanym kryptosystem z kluczem publicznym jest RSA (nazwa pochodzi od wynalazów: Rivest, Shamir i Adlemann). RSA umożliwia szyfrowanie danych jak również realizację podpisu cyfrowego. Bezpieczeństwo RSA opiera się na obliczeniowej trudności rozwiązania **problemu faktoryzacji liczb całkowitych złożonych**. 

## Generowanie kluczy w kryptosystemie RSA

### 1. Losujemy dwie duże liczby pierwsze 
Potrzebujemy dwóch liczb pierwszych o naprawdę dużych rozmiarach - 2048 bitów obecnie uważa się niezbyt bezpieczny wybór. 4096 bitów jest z kolei wielkością nieco kłopotliwą w użytkowaniu. 
#### Skąd wziąć liczbę pierwszą? 
**Wylosować i sprawdzić czy jest pierwsza!**


Test probabilistyczny, np. Rabina-Millera. **(A to już znamy!!!)**

## Zadanie
1. Napisz funkcję generującą liczbę pierwszą o określonej długości w bitach. 

In [7]:
import secrets

def is_probable_prime(n, k=40):
    if n < 2:
        return False
    small_primes = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
    for p in small_primes:
        if n % p == 0:
            return n == p
    r = 0
    d = n - 1
    while d % 2 == 0:
        d //= 2
        r += 1
    for _ in range(k):
        a = secrets.randbelow(n - 3) + 2
        x = pow(a, d, n)
        if x == 1 or x == n - 1:
            continue
        for _ in range(r - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False
    return True

def generatePrime(keysize):
    while True:
        num = secrets.randbits(keysize)
        num |= (1 << (keysize - 1)) | 1
        if is_probable_prime(num):
            return num

p = generatePrime(256)
print("Wygenerowano liczbę pierwszą:", p, "bitów:", p.bit_length())

### 2. Obliczamy składniki kluczy 
1. Wybieramy dwie duże liczby pierwsze $p$ i $q$
2. Pierwszym składnikiem klucza jest moduł $n$ $n=p \times q$ 
3. Poszukujemy wykładnika publicznego $e$, który jest względnie pierwszy z $(p-1)\cdot (q-1)$ (czasami używane jest w miejscu pojęcie tocjentu lub funkcji Eulera: $\phi(n) = \phi(p)\cdot \phi(q) = (p − 1)·(q − 1)$)
4. Poszukujemy wykładnika prywatnego $d$, które jest odwrotnością $e\ (mod\ (p-1)\cdot (q-1))$: $de \equiv 1  (mod\ (p-1)\cdot (q-1))$ **(potrzebujemy rozszerzonego algorytmu Euklidesa!!!)**
5. Kluczem publiczny jest para $(n, e)$, kluczem prywatnym jest para $(n, d)$.

## Zadanie 

1. Napisz funkcję generującą klucze RSA o ustalonym rozmiarze

In [8]:
import math

def generateRSAKeys(keysize, e=65537):
    """Zwraca ((n, e), (n, d), p, q) — klucz publiczny, prywatny i składniki do CRT."""
    while True:
        p = generatePrime(keysize // 2)
        q = generatePrime(keysize // 2)
        if p == q:
            continue
        n = p * q
        phi = (p - 1) * (q - 1)
        if math.gcd(e, phi) != 1:
            continue
        d = pow(e, -1, phi)
        return (n, e), (n, d), p, q

(n, e), (n, d), p, q = generateRSAKeys(256)
m = 42
c = pow(m, e, n)
assert pow(c, d, n) == m
print("Klucze RSA OK, moduł ma", n.bit_length(), "bitów")

## Zadanie 

Napisz funkcje implementujące szyfrowanie i deszyfrowanie RSA (tzw. podręcznikowe)

### Szyfrowanie RSA 
Operacja szyfrowania: $c=m^e (mod\ n)$

In [9]:
def encrypt(message, modulus, exp):
    """
    Textbook RSA encryption.
    - If message is int -> returns int ciphertext.
    - If message is str/bytes -> returns int (if fits) or list[int] (blocked).
    """
    # handle integer message
    if isinstance(message, int):
        if message >= modulus:
            raise ValueError("Integer message must be < modulus")
        return pow(message, exp, modulus)

    # normalize to bytes
    if isinstance(message, str):
        m_bytes = message.encode('utf-8')
    elif isinstance(message, bytes):
        m_bytes = message
    else:
        raise TypeError("message must be int, str or bytes")

    m_int = int.from_bytes(m_bytes, 'big')
    if m_int < modulus:
        return pow(m_int, exp, modulus)

    # split into blocks so each block as integer < modulus
    k = (modulus.bit_length() - 1) // 8  # max bytes per block
    if k <= 0:
        raise ValueError("modulus too small for blocking")
    blocks = [m_bytes[i:i + k] for i in range(0, len(m_bytes), k)]
    cipher_blocks = [pow(int.from_bytes(b, 'big'), exp, modulus) for b in blocks]
    return cipher_blocks

### Deszyfrowanie RSA 
Operacja szyfrowanie $m = c^d (mod\ n)$

In [10]:
def decrypt(message_encrypted, modulus, exp):
    """
    Textbook RSA decryption.
    - If message_encrypted is int -> returns str.
    - If list/tuple of ints -> returns str (joined blocks).
    """
    def _int_to_bytes(i):
        if i == 0:
            return b'\x00'
        return i.to_bytes((i.bit_length() + 7) // 8, 'big')

    if isinstance(message_encrypted, int):
        m_int = pow(message_encrypted, exp, modulus)
        return _int_to_bytes(m_int).decode('utf-8', errors='ignore')

    if isinstance(message_encrypted, (list, tuple)):
        parts = []
        for c in message_encrypted:
            if not isinstance(c, int):
                raise TypeError("cipher blocks must be integers")
            m_int = pow(c, exp, modulus)
            parts.append(_int_to_bytes(m_int))
        return b''.join(parts).decode('utf-8', errors='ignore')

    raise TypeError("message_encrypted must be int or list/tuple of ints")

In [ ]:
(n, e), (n, d), _, _ = generateRSAKeys(512)
msg = "Kryptografia RSA"
c = encrypt(msg, n, e)
print("Odszyfrowano:", decrypt(c, n, d))
print("OK:", decrypt(c, n, d) == msg)

## Zastanów się
1. Sprawdź działanie powyższej implementacji dla różnych wielkości klucza (podawane podczas generowania kluczy) - zweryfikuj jak na wydajnosć wpływa zastosowanie różnych sposobów potęgowania dostępnych w Python i jego bibliotekach.  
2. Poszukaj informacji o trybie podręcznikowym RSA (*textbook RSA encryption*). Na czym polega? Jakie są jego wady i zalety? 


In [11]:
import time, secrets
from math import log2

def bench_once(modulus, exp, msg_int, repeat=50):
    t0 = time.perf_counter()
    for _ in range(repeat):
        pow(msg_int, exp, modulus)
    return (time.perf_counter() - t0)/repeat

def rsa_crt_decrypt(c, p, q, d):
    dp = d % (p-1); dq = d % (q-1)
    q_inv = pow(q, -1, p)
    m1 = pow(c, dp, p); m2 = pow(c, dq, q)
    h = (m1 - m2) * q_inv % p
    return m2 + q * h

for bits in (512, 1024, 2048):
    (n, e), (n, d), p, q = generateRSAKeys(bits)
    m = secrets.randbelow(n-1)+1
    c = pow(m, e, n)
    print(bits, "enc", bench_once(n, e, m))
    t_plain = bench_once(n, d, c)
    t0 = time.perf_counter()
    for _ in range(50): rsa_crt_decrypt(c,p,q,d)
    t_crt = (time.perf_counter()-t0)/50
    print(bits, "dec plain", t_plain, "dec CRT", t_crt)

512 enc 3.060200000618352e-05
512 dec plain 0.001754854000000705 dec CRT 0.0008024420000037935
1024 enc 0.0001386800000000221
1024 dec plain 0.00835402200000317 dec CRT 0.003970190000000002
2048 enc 0.00018482599999515516
2048 dec plain 0.03216854199999943 dec CRT 0.009046363999996174


### Algorytm szybkiego potęgowania 
1. Zwykłe potęgowanie $n^{exp}$: $exp$ mnożeń 
2. Algorytm szybkiego potęgowania: część mnożeń zastępujemy podnoszeniem do kwadratu (_squaring_).
    __Skąd mamy wiedzieć kiedy mnożyć, a kiedy potęgować?__

In [12]:

def fastModularExponentation(b, exp, m):
    res = 1
    while exp > 1:
        if exp & 1:
            res = (res * b) % m
        b = b ** 2 % m
        exp >>= 1
    return (b * res) % m